# Bronze Customer Ingestion

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
file_path = "/Volumes/ecommerce_lakehouse/raw/oltp_landing/customers/olist_customers_dataset.csv"

column_names = spark.read \
                .format("csv") \
                .option("header","True") \
                .load(file_path) \
                .limit(5)

display(column_names)

In [0]:
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True)
])

In [0]:
customers_stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema(customers_schema)
        .load(
            "/Volumes/ecommerce_lakehouse/raw/oltp_landing/customers/"
        )
)

In [0]:
bronze_customers_df = customers_stream_df \
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    ) \
    .withColumn(
        "load_date",
        current_date()
    ) \
    .withColumn(
        "source_file",
        col("_metadata.file_path")
    )

In [0]:
query = (
    bronze_customers_df.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "/Volumes/ecommerce_lakehouse/raw/checkpoints/customers/"
        )
        .trigger(availableNow=True)
        .toTable(
            "ecommerce_lakehouse.bronze.customers_raw"
        )
)

In [0]:
%sql
SELECT *
FROM ecommerce_lakehouse.bronze.customers_raw
LIMIT 10;